# Barter-RS: Custom Trading Strategies

This notebook explains the Strategy trait system and shows how to implement
custom trading strategies, including multi-strategy composition.

## Topics Covered
- The 4 strategy traits: `AlgoStrategy`, `ClosePositionsStrategy`, `OnDisconnectStrategy`, `OnTradingDisabled`
- Implementing a simple strategy
- Multi-strategy composition with per-strategy position tracking
- Custom instrument data for strategy-specific state
- Using `StrategyId` to tag orders by strategy

In [ ]:
:dep barter = { version = "0.12" }
:dep barter-data = { version = "0.11" }
:dep barter-execution = { version = "0.7" }
:dep barter-instrument = { version = "0.3" }
:dep barter-integration = { version = "0.10" }
:dep rust_decimal = { version = "1.36", features = ["maths"] }
:dep rust_decimal_macros = "1.29"
:dep smol_str = "0.3"
:dep chrono = { version = "0.4", features = ["serde"] }
:dep itertools = "0.14"

## 1. Strategy Trait Overview

Every strategy must implement 4 traits. Here's what each one does:

| Trait | When Called | Purpose |
|-------|------------|---------|
| `AlgoStrategy` | Every engine tick (market data or account event) | Generate order requests (opens & cancels) |
| `ClosePositionsStrategy` | On shutdown or explicit close command | Generate orders to flatten all open positions |
| `OnDisconnectStrategy` | When an exchange WebSocket disconnects | React to connectivity loss (e.g., cancel orders) |
| `OnTradingDisabled` | When trading state transitions to Disabled | Clean up when trading is turned off |

For simple strategies, `OnDisconnectStrategy` and `OnTradingDisabled` can be no-ops.

In [ ]:
use barter::{
    engine::{
        Engine, Processor,
        state::{
            EngineState,
            global::DefaultGlobalData,
            instrument::{
                data::{DefaultInstrumentMarketData, InstrumentDataState},
                filter::InstrumentFilter,
            },
            order::in_flight_recorder::InFlightRequestRecorder,
        },
    },
    strategy::{
        algo::AlgoStrategy,
        close_positions::{ClosePositionsStrategy, build_ioc_market_order_to_close_position},
        on_disconnect::OnDisconnectStrategy,
        on_trading_disabled::OnTradingDisabled,
    },
};
use barter_data::event::{DataKind, MarketEvent};
use barter_execution::{
    AccountEvent,
    order::{
        id::{ClientOrderId, StrategyId},
        request::{OrderRequestCancel, OrderRequestOpen},
    },
};
use barter_instrument::{
    asset::AssetIndex,
    exchange::{ExchangeId, ExchangeIndex},
    instrument::InstrumentIndex,
};
use rust_decimal::Decimal;
use smol_str::SmolStr;

println!("Strategy traits imported.");

## 2. Minimal Strategy Implementation

The simplest possible strategy that does nothing (equivalent to `DefaultStrategy`).
This is a good starting template.

In [ ]:
/// A no-op strategy that never generates orders.
#[derive(Debug, Default)]
struct NoOpStrategy;

// Core: generate order requests each tick
impl AlgoStrategy for NoOpStrategy {
    type State = EngineState<DefaultGlobalData, DefaultInstrumentMarketData>;

    fn generate_algo_orders(
        &self,
        _state: &Self::State, // full engine state is available here
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        // Return empty iterators = no orders
        (std::iter::empty(), std::iter::empty())
    }
}

// Shutdown: generate orders to close all positions
impl ClosePositionsStrategy for NoOpStrategy {
    type State = EngineState<DefaultGlobalData, DefaultInstrumentMarketData>;

    fn close_positions_requests<'a>(
        &'a self,
        state: &'a Self::State,
        filter: &'a InstrumentFilter,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel> + 'a,
        impl IntoIterator<Item = OrderRequestOpen> + 'a,
    )
    where
        ExchangeIndex: 'a,
        AssetIndex: 'a,
        InstrumentIndex: 'a,
    {
        // No positions to close in this simple strategy
        (std::iter::empty(), std::iter::empty())
    }
}

// Connectivity: react to exchange disconnections
impl<Clock, State, ExecutionTxs, Risk> OnDisconnectStrategy<Clock, State, ExecutionTxs, Risk>
    for NoOpStrategy
{
    type OnDisconnect = ();

    fn on_disconnect(
        _engine: &mut Engine<Clock, State, ExecutionTxs, Self, Risk>,
        _exchange: ExchangeId,
    ) -> Self::OnDisconnect {
        // Could cancel open orders on the disconnected exchange here
    }
}

// Trading control: react when trading is disabled
impl<Clock, State, ExecutionTxs, Risk> OnTradingDisabled<Clock, State, ExecutionTxs, Risk>
    for NoOpStrategy
{
    type OnTradingDisabled = ();

    fn on_trading_disabled(
        _engine: &mut Engine<Clock, State, ExecutionTxs, Self, Risk>,
    ) -> Self::OnTradingDisabled {
        // Could flush state, cancel orders, etc.
    }
}

println!("NoOpStrategy defined with all 4 required trait implementations.");

## 3. Multi-Strategy Composition

A common pattern is running multiple sub-strategies in a single engine.
Each sub-strategy gets its own `StrategyId` and position tracking.

The key idea:
1. Define a `MultiStrategy` struct containing sub-strategies
2. Create custom `InstrumentData` that tracks per-strategy positions
3. Implement `AlgoStrategy` by chaining sub-strategy outputs
4. Use `StrategyId` to tag orders so fills route to the correct sub-strategy

In [ ]:
use barter::engine::state::position::PositionManager;
use barter::statistic::summary::instrument::TearSheetGenerator;
use barter_execution::AccountEventKind;
use chrono::{DateTime, Utc};

/// Multi-strategy wrapper that composes StrategyA and StrategyB
#[derive(Debug, Default)]
struct MultiStrategy {
    strategy_a: StrategyA,
    strategy_b: StrategyB,
}

#[derive(Debug, Default)]
struct StrategyA;
impl StrategyA {
    const ID: StrategyId = StrategyId(SmolStr::new_static("strategy_a"));
}

#[derive(Debug, Default)]
struct StrategyB;
impl StrategyB {
    const ID: StrategyId = StrategyId(SmolStr::new_static("strategy_b"));
}

/// Per-strategy instrument data: each sub-strategy tracks its own positions
#[derive(Debug, Clone)]
struct PerStrategyData {
    tear: TearSheetGenerator,
    position: PositionManager,
}

/// Custom instrument data combining market data + per-strategy state
#[derive(Debug, Clone)]
struct MultiStrategyInstrumentData {
    market_data: DefaultInstrumentMarketData,
    strategy_a: PerStrategyData,
    strategy_b: PerStrategyData,
}

println!("Multi-strategy types defined.");
println!("  StrategyA ID: {}", StrategyA::ID.0);
println!("  StrategyB ID: {}", StrategyB::ID.0);

## 4. Implementing Traits for Custom Instrument Data

Custom instrument data must implement several traits so the engine can
process events through it:

In [ ]:
// InstrumentDataState: exposes the current market price
impl InstrumentDataState for MultiStrategyInstrumentData {
    type MarketEventKind = DataKind;

    fn price(&self) -> Option<Decimal> {
        self.market_data.price()
    }
}

// Process market events: delegate to inner DefaultInstrumentMarketData
impl<InstrumentKey> Processor<&MarketEvent<InstrumentKey, DataKind>>
    for MultiStrategyInstrumentData
{
    type Audit = ();

    fn process(&mut self, event: &MarketEvent<InstrumentKey, DataKind>) -> Self::Audit {
        self.market_data.process(event)
    }
}

// Process account events: route trades to the correct sub-strategy by StrategyId
impl Processor<&AccountEvent> for MultiStrategyInstrumentData {
    type Audit = ();

    fn process(&mut self, event: &AccountEvent) -> Self::Audit {
        let AccountEventKind::Trade(trade) = &event.kind else {
            return;
        };

        // Route trade to the correct strategy based on StrategyId
        if trade.strategy == StrategyA::ID {
            self.strategy_a
                .position
                .update_from_trade(trade)
                .inspect(|closed| self.strategy_a.tear.update_from_position(closed));
        }

        if trade.strategy == StrategyB::ID {
            self.strategy_b
                .position
                .update_from_trade(trade)
                .inspect(|closed| self.strategy_b.tear.update_from_position(closed));
        }
    }
}

// Track in-flight order requests (for duplicate detection)
impl InFlightRequestRecorder for MultiStrategyInstrumentData {
    fn record_in_flight_cancel(&mut self, _: &OrderRequestCancel<ExchangeIndex, InstrumentIndex>) {}
    fn record_in_flight_open(&mut self, _: &OrderRequestOpen<ExchangeIndex, InstrumentIndex>) {}
}

println!("All required trait impls defined for MultiStrategyInstrumentData.");

## 5. Composing AlgoStrategy

The `MultiStrategy` chains outputs from both sub-strategies:

In [ ]:
type MultiState = EngineState<DefaultGlobalData, MultiStrategyInstrumentData>;

// Each sub-strategy implements AlgoStrategy independently
impl AlgoStrategy for StrategyA {
    type State = MultiState;
    fn generate_algo_orders(
        &self, _: &Self::State,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        // Your strategy logic here — access state.instruments, state.assets, etc.
        (std::iter::empty(), std::iter::empty())
    }
}

impl AlgoStrategy for StrategyB {
    type State = MultiState;
    fn generate_algo_orders(
        &self, _: &Self::State,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        (std::iter::empty(), std::iter::empty())
    }
}

// Composite: chain both sub-strategies' outputs
impl AlgoStrategy for MultiStrategy {
    type State = MultiState;

    fn generate_algo_orders(
        &self, state: &Self::State,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        let (cancels_a, opens_a) = self.strategy_a.generate_algo_orders(state);
        let (cancels_b, opens_b) = self.strategy_b.generate_algo_orders(state);

        // Chain all cancel and open requests from both strategies
        (cancels_a.into_iter().chain(cancels_b), opens_a.into_iter().chain(opens_b))
    }
}

println!("MultiStrategy composes StrategyA + StrategyB via iterator chaining.");

## Strategy Architecture Diagram

```
┌─────────────────────────────────────────────────────┐
│                    MultiStrategy                     │
│                                                     │
│  ┌─────────────┐         ┌─────────────┐           │
│  │ StrategyA    │         │ StrategyB    │           │
│  │ ID: "strat_a"│         │ ID: "strat_b"│           │
│  └──────┬───────┘         └──────┬───────┘           │
│         │                        │                   │
│         ▼                        ▼                   │
│   (cancels_a,              (cancels_b,               │
│    opens_a)                 opens_b)                 │
│         │                        │                   │
│         └────────┬───────────────┘                   │
│                  ▼                                   │
│         .chain() both iterators                      │
└─────────────────────────────────────────────────────┘
                   │
                   ▼
           ┌───────────────┐
           │  RiskManager   │  ← approve/reject
           └───────┬───────┘
                   ▼
           ┌───────────────┐
           │  Execution     │  ← submit to exchange
           └───────────────┘
```

### Key Design Points

- **`StrategyId`**: Each sub-strategy tags its orders so trade fills route back correctly
- **`PositionManager`**: Per-strategy position tracking (not shared across strategies)
- **`TearSheetGenerator`**: Per-strategy performance statistics
- **`InstrumentDataState`**: Custom data must expose `price()` for the engine to use
- **Composition over inheritance**: Strategies are composed via iterator chaining, not trait hierarchy